# Evaluación de LLMs

> Aprende a evaluar la calidad de respuestas de modelos de lenguaje usando LLM-as-Judge con Claude,
> métricas automáticas como BLEU, y visualización de resultados con pandas.

## Objetivos
- Implementar un juez LLM que evalúa respuestas con una rúbrica estructurada
- Evaluar un conjunto de respuestas de ejemplo
- Calcular métricas BLEU básicas con NLTK
- Visualizar resultados con pandas

## 1. Instalación de dependencias

In [ ]:
%pip install anthropic nltk pandas matplotlib --quiet

## 2. Configuración inicial

In [ ]:
import anthropic
import json
import pandas as pd
import matplotlib.pyplot as plt
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import os

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Inicializar cliente de Anthropic
# La API key se lee de la variable de entorno ANTHROPIC_API_KEY
client = anthropic.Anthropic()
MODELO = "claude-haiku-4-5-20251001"

print(f"Cliente inicializado. Modelo: {MODELO}")

## 3. Función LLM-as-Judge

Implementamos un juez que evalúa respuestas usando una rúbrica JSON estructurada.
El juez devuelve puntuaciones en varias dimensiones y una justificación.

In [ ]:
RUBRICA = """
Evalúa la respuesta del asistente según los siguientes criterios (0-10 cada uno):

1. precision: ¿La respuesta es factualmente correcta?
2. completitud: ¿Responde completamente la pregunta?
3. claridad: ¿Es clara y fácil de entender?
4. concision: ¿Es concisa sin perder información importante?
5. utilidad: ¿Es útil para quien hace la pregunta?

Devuelve ÚNICAMENTE un JSON con esta estructura:
{
  "precision": <0-10>,
  "completitud": <0-10>,
  "claridad": <0-10>,
  "concision": <0-10>,
  "utilidad": <0-10>,
  "justificacion": "<explicación breve>",
  "puntuacion_total": <promedio>
}
"""

def evaluar_respuesta(pregunta: str, respuesta: str) -> dict:
    """Evalúa una respuesta usando Claude como juez."""
    prompt = f"""Pregunta del usuario: {pregunta}

Respuesta a evaluar: {respuesta}

{RUBRICA}"""

    mensaje = client.messages.create(
        model=MODELO,
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )

    texto = mensaje.content[0].text.strip()
    # Limpiar posibles bloques de código markdown
    if texto.startswith("```"):
        texto = texto.split("```")[1]
        if texto.startswith("json"):
            texto = texto[4:]
    return json.loads(texto)

# Prueba rápida
resultado_prueba = evaluar_respuesta(
    pregunta="¿Qué es Python?",
    respuesta="Python es un lenguaje de programación de alto nivel, interpretado y de propósito general."
)
print("Resultado de prueba:")
print(json.dumps(resultado_prueba, indent=2, ensure_ascii=False))

## 4. Evaluación de 5 respuestas de ejemplo

Evaluamos respuestas de calidad variable para la misma pregunta.

In [ ]:
# Conjunto de evaluación: pregunta + respuestas de calidad variable
evaluaciones = [
    {
        "id": 1,
        "pregunta": "¿Cómo funciona el aprendizaje automático?",
        "respuesta": "El aprendizaje automático es un subcampo de la IA donde los sistemas aprenden patrones "
                    "a partir de datos sin ser programados explícitamente. Utiliza algoritmos como regresión, "
                    "árboles de decisión y redes neuronales para hacer predicciones o clasificaciones.",
        "tipo": "buena"
    },
    {
        "id": 2,
        "pregunta": "¿Cómo funciona el aprendizaje automático?",
        "respuesta": "Las computadoras aprenden solas.",
        "tipo": "muy_corta"
    },
    {
        "id": 3,
        "pregunta": "¿Cómo funciona el aprendizaje automático?",
        "respuesta": "El aprendizaje automático (Machine Learning) funciona mediante tres etapas principales: "
                    "1) Recolección y preparación de datos de entrenamiento, 2) Selección y entrenamiento "
                    "de un modelo matemático que ajusta sus parámetros minimizando el error, y "
                    "3) Evaluación y despliegue del modelo para hacer predicciones sobre datos nuevos. "
                    "Existen tres paradigmas: supervisado (con etiquetas), no supervisado (sin etiquetas) "
                    "y por refuerzo (con recompensas).",
        "tipo": "excelente"
    },
    {
        "id": 4,
        "pregunta": "¿Cómo funciona el aprendizaje automático?",
        "respuesta": "El aprendizaje automático usa matemáticas y estadística. Los algoritmos procesan "
                    "números en matrices. Hay backpropagation y gradient descent. También existe el "
                    "overfitting que hay que evitar con regularización L1 y L2 o dropout.",
        "tipo": "tecnica_sin_contexto"
    },
    {
        "id": 5,
        "pregunta": "¿Cómo funciona el aprendizaje automático?",
        "respuesta": "El aprendizaje automático permite que los sistemas mejoren con la experiencia. "
                    "Se alimenta al modelo con ejemplos históricos y aprende a reconocer patrones, "
                    "similar a cómo los humanos aprendemos de la práctica.",
        "tipo": "analogia_simple"
    }
]

print(f"Evaluando {len(evaluaciones)} respuestas...\n")
resultados = []

for ev in evaluaciones:
    print(f"Evaluando respuesta #{ev['id']} (tipo: {ev['tipo']})...")
    evaluacion = evaluar_respuesta(ev["pregunta"], ev["respuesta"])
    resultados.append({
        "id": ev["id"],
        "tipo": ev["tipo"],
        **evaluacion
    })

print("\n¡Evaluación completada!")

## 5. Visualización de resultados con pandas

In [ ]:
# Crear DataFrame con resultados
df = pd.DataFrame(resultados)
columnas_metricas = ["precision", "completitud", "claridad", "concision", "utilidad", "puntuacion_total"]

print("=== Resultados de Evaluación LLM-as-Judge ===")
print(df[["id", "tipo"] + columnas_metricas].to_string(index=False))

print("\n=== Estadísticas por métrica ===")
print(df[columnas_metricas].describe().round(2))

# Gráfica de radar / barras por respuesta
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfica 1: Puntuación total por tipo de respuesta
ax1 = axes[0]
colores = ["#2ecc71", "#e74c3c", "#3498db", "#f39c12", "#9b59b6"]
bars = ax1.bar(df["tipo"], df["puntuacion_total"], color=colores, edgecolor="black", alpha=0.8)
ax1.set_title("Puntuación Total por Tipo de Respuesta", fontweight="bold")
ax1.set_xlabel("Tipo de Respuesta")
ax1.set_ylabel("Puntuación (0-10)")
ax1.set_ylim(0, 10)
ax1.tick_params(axis="x", rotation=30)
for bar, val in zip(bars, df["puntuacion_total"]):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f"{val:.1f}", ha="center", va="bottom", fontweight="bold")

# Gráfica 2: Heatmap de métricas
ax2 = axes[1]
metricas_df = df[["tipo"] + ["precision", "completitud", "claridad", "concision", "utilidad"]].set_index("tipo")
im = ax2.imshow(metricas_df.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=10)
ax2.set_xticks(range(len(metricas_df.columns)))
ax2.set_xticklabels(metricas_df.columns, rotation=30)
ax2.set_yticks(range(len(metricas_df.index)))
ax2.set_yticklabels(metricas_df.index)
ax2.set_title("Mapa de Calor de Métricas", fontweight="bold")
plt.colorbar(im, ax=ax2, label="Puntuación")
for i in range(len(metricas_df.index)):
    for j in range(len(metricas_df.columns)):
        ax2.text(j, i, f"{metricas_df.values[i,j]:.0f}",
                ha="center", va="center", fontweight="bold")

plt.tight_layout()
plt.savefig("evaluacion_resultados.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfica guardada como 'evaluacion_resultados.png'")

## 6. Métricas BLEU básicas con NLTK

BLEU (Bilingual Evaluation Understudy) mide la similitud n-gram entre una respuesta
generada y una respuesta de referencia. Es una métrica clásica en NLP.

In [ ]:
from nltk.tokenize import word_tokenize

# Respuesta de referencia ("gold standard")
REFERENCIA = """El aprendizaje automático es un subcampo de la inteligencia artificial que permite 
a los sistemas aprender y mejorar a partir de la experiencia sin ser programados explícitamente. 
Los algoritmos aprenden patrones de datos de entrenamiento para hacer predicciones o decisiones."""

referencia_tokens = word_tokenize(REFERENCIA.lower())

def calcular_bleu(referencia_tokens: list, hipotesis: str) -> dict:
    """Calcula métricas BLEU 1-4 para una hipótesis."""
    hipotesis_tokens = word_tokenize(hipotesis.lower())
    smooth = SmoothingFunction().method1

    bleu1 = sentence_bleu([referencia_tokens], hipotesis_tokens,
                          weights=(1, 0, 0, 0), smoothing_function=smooth)
    bleu2 = sentence_bleu([referencia_tokens], hipotesis_tokens,
                          weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    bleu4 = sentence_bleu([referencia_tokens], hipotesis_tokens,
                          weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)
    return {"BLEU-1": round(bleu1, 4), "BLEU-2": round(bleu2, 4), "BLEU-4": round(bleu4, 4)}

print("=== Métricas BLEU por Respuesta ===")
print(f"Referencia: '{REFERENCIA[:80]}...'\n")

bleu_resultados = []
for ev in evaluaciones:
    bleu = calcular_bleu(referencia_tokens, ev["respuesta"])
    bleu_resultados.append({"tipo": ev["tipo"], **bleu})
    print(f"Tipo: {ev['tipo']:25s} | BLEU-1: {bleu['BLEU-1']:.4f} | BLEU-2: {bleu['BLEU-2']:.4f} | BLEU-4: {bleu['BLEU-4']:.4f}")

# Combinar con resultados del juez
df_bleu = pd.DataFrame(bleu_resultados)
df_final = df.merge(df_bleu, on="tipo")

print("\n=== Correlación entre BLEU y puntuación del juez ===")
correlacion = df_final[["puntuacion_total", "BLEU-1", "BLEU-2", "BLEU-4"]].corr()
print(correlacion.round(3))
print("\nNota: BLEU mide similitud léxica, no semántica. El juez LLM captura mejor la calidad real.")